In [2]:
from manim import *

config.media_width = "75%"
config.verbosity = "WARNING"

Manim Community v0.18.0.post0

In [3]:
class L1Regularization(Scene):
    def construct(self):
        # Title
        title = Text("L1 Regularization Effect on Sparsity")
        self.play(Write(title))
        self.wait(1)
        self.play(FadeOut(title))

        # Setting up the weight matrix visualization as a grid
        grid_size = 5  # 5x5 grid for simplicity
        matrix_elements = VGroup()
        
        # Initialize weights as random numbers in a small range
        initial_weights = np.random.uniform(-1, 1, (grid_size, grid_size))

        # Display initial weights
        for i in range(grid_size):
            for j in range(grid_size):
                elem = DecimalNumber(initial_weights[i, j], num_decimal_places=2)
                elem.move_to((i - 2, j - 2, 0))
                matrix_elements.add(elem)
        
        self.play(Create(matrix_elements))
        self.wait(1)

        # Apply L1 regularization effect over epochs
        epochs = 5
        for epoch in range(epochs):
            for i in range(len(matrix_elements)):
                weight = matrix_elements[i].get_value()
                # Reduce weight by a small constant factor to simulate L1 regularization
                updated_weight = max(0, abs(weight) - 0.1) * np.sign(weight)
                matrix_elements[i].set_value(updated_weight)
            
            # Add epoch label
            epoch_text = Text(f"Epoch {epoch + 1}", font_size=24)
            epoch_text.to_edge(UP)
            self.play(Transform(matrix_elements, matrix_elements), Write(epoch_text))
            self.wait(0.5)
            self.play(FadeOut(epoch_text))

        # Final message
        final_text = Text("L1 Regularization Drives Weights to Zero (Sparsity)", font_size=24)
        final_text.to_edge(DOWN)
        self.play(Write(final_text))
        self.wait(2)
%manim -qm L1Regularization

In [5]:
from manim import *
import numpy as np
import random

class L1RegularizationMNIST(Scene):
    def construct(self):
        # Title
        title = Text("L1 Regularization and Sparse Gradients")
        self.play(Write(title))
        self.wait(1)
        self.play(FadeOut(title))

        # Display a few sample images (representing MNIST digits)
        digit_texts = ["3", "7", "2"]
        mnist_images = VGroup()
        for i, digit in enumerate(digit_texts):
            mnist_image = Text(digit, font_size=72)
            mnist_image.move_to((i * 2.5 - 2.5, 1.5, 0))
            mnist_images.add(mnist_image)
        
        self.play(FadeIn(mnist_images))
        self.wait(1)

        # Define initial weights for a small neural network layer
        grid_size = 5  # Simplified 5x5 weight matrix
        weights = np.random.uniform(-0.5, 0.5, (grid_size, grid_size))
        
        # Display weight matrix with values
        weight_matrix = VGroup()
        for i in range(grid_size):
            for j in range(grid_size):
                weight_value = DecimalNumber(weights[i, j], num_decimal_places=2)
                weight_value.move_to((i - 2, j - 2, 0))
                weight_matrix.add(weight_value)
        
        self.play(Create(weight_matrix))
        self.wait(1)

        # Function to apply L1 regularization effect
        def apply_l1_step(weights):
            new_weights = weights.copy()
            for i in range(len(weights)):
                for j in range(len(weights[0])):
                    gradient = np.sign(weights[i, j]) * 0.1
                    new_weights[i, j] -= gradient  # L1 regularization update step
                    # Zero out smaller weights to emphasize sparsity
                    if abs(new_weights[i, j]) < 0.05:
                        new_weights[i, j] = 0
            return new_weights

        # Epoch simulation with weight updates
        epochs = 5
        for epoch in range(epochs):
            weights = apply_l

%manim -qm L1RegularizationMNIST

NameError: name 'apply_l' is not defined

In [7]:
from manim import *
import numpy as np

class NormVisualization(Scene):
    def construct(self):
        # Create coordinate system
        axes = Axes(
            x_range=[0, 10, 1],
            y_range=[0, 10, 1],
            axis_config={"include_tip": True},
            # x_axis_config={"label_text": "Square Footage (1000s)"},
            # y_axis_config={"label_text": "Price (100k $)"}
        )
        axes_labels = axes.get_axis_labels()

        # Sample data points
        data_points = [
            [2, 3], [3, 4], [4, 4.5], [5, 5.5],
            [6, 6], [7, 7.2], [8, 7.8]
        ]
        
        # Create dots for data points
        dots = VGroup(*[Dot(axes.c2p(x, y)) for x, y in data_points])
        
        # Best fit line
        best_fit = LinesToReg(
            list(map(lambda x: axes.c2p(*x), data_points)),
            color=BLUE
        )

        # Initial setup
        self.play(
            Create(axes),
            Write(axes_labels),
            FadeIn(dots)
        )
        self.wait()

        # L1 Norm Visualization
        l1_lines = self.create_error_lines(axes, data_points, best_fit, "L1")
        self.play(Create(best_fit))
        self.play(*[Create(line) for line in l1_lines])
        self.wait()
        
        # Transition to L2 Norm
        l2_lines = self.create_error_lines(axes, data_points, best_fit, "L2")
        self.play(
            *[Transform(l1, l2) for l1, l2 in zip(l1_lines, l2_lines)]
        )
        self.wait()
        
        # Transition to L-infinity Norm
        linf_lines = self.create_error_lines(axes, data_points, best_fit, "Linf")
        self.play(
            *[Transform(l2, linf) for l2, linf in zip(l2_lines, linf_lines)]
        )
        self.wait()

    def create_error_lines(self, axes, data_points, best_fit, norm_type):
        lines = VGroup()
        for point in data_points:
            x, y = point
            # Get y-value on best fit line
            best_fit_y = self.get_best_fit_y(x)
            
            if norm_type == "L1":
                # Vertical lines for L1 norm
                line = Line(
                    start=axes.c2p(x, y),
                    end=axes.c2p(x, best_fit_y),
                    color=RED,
                    stroke_width=2
                )
            elif norm_type == "L2":
                # Perpendicular lines for L2 norm
                slope = -1/best_fit.get_slope()
                dx = (best_fit_y - y) / np.sqrt(1 + slope**2)
                line = Line(
                    start=axes.c2p(x, y),
                    end=axes.c2p(x + dx, y + slope*dx),
                    color=GREEN,
                    stroke_width=2
                )
            else:  # L-infinity
                # Horizontal lines for L-infinity norm
                max_error = max(abs(best_fit_y - y) for x, y in data_points)
                line = Line(
                    start=axes.c2p(x, y),
                    end=axes.c2p(x, y + max_error),
                    color=YELLOW,
                    stroke_width=2
                )
            lines.add(line)
        return lines

    def get_best_fit_y(self, x):
        # Simple linear relationship for demonstration
        return 0.8 * x + 1.5

class LinesToReg(Line):
    def __init__(self, points, **kwargs):
        x_coords = [p[0] for p in points]
        y_coords = [p[1] for p in points]
        x_mean = np.mean(x_coords)
        y_mean = np.mean(y_coords)
        
        # Calculate slope and intercept
        numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(x_coords, y_coords))
        denominator = sum((x - x_mean)**2 for x in x_coords)
        slope = numerator / denominator
        intercept = y_mean - slope * x_mean
        
        # Create line points
        x_min, x_max = min(x_coords), max(x_coords)
        start_point = np.array([x_min, slope * x_min + intercept, 0])
        end_point = np.array([x_max, slope * x_max + intercept, 0])
        
        super().__init__(start=start_point, end=end_point, **kwargs)
        self.slope = slope
        self.intercept = intercept

    def get_slope(self):
        return self.slope
    
%manim -qm NormVisualization

In [10]:
from manim import *
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Function to simulate a simple gradient update step
def l1_gradient_update(weights, gradients, lr=0.1, reg_strength=0.01):
    return weights - lr * (gradients + reg_strength * np.sign(weights))

class L1RegularizationWithMNIST(Scene):
    def construct(self):
        # Load MNIST-like data (digits dataset as a proxy)
        digits = load_digits()
        X = digits.data
        y = digits.target
        X = StandardScaler().fit_transform(X)  # Normalize
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Use a small subset for simplicity
        X_train = X_train[:5]
        y_train = y_train[:5]

        # Simple neural network parameters (1 layer)
        input_size = X_train.shape[1]  # 64 features per digit
        output_size = len(np.unique(y))  # 10 classes for MNIST digits
        weights = np.random.uniform(-1, 1, (input_size, output_size))

        # Create a visualization of input digits as circles (representing pixels)
        digit_images = VGroup()
        for i, digit in enumerate(X_train):
            digit_img = VGroup()
            # Represent each feature as a small circle, scaled by the feature's value
            for j, feature_value in enumerate(digit):
                # Create circles with radius proportional to feature value (scaled)
                circle = Circle(radius=0.1 + 0.05 * feature_value)
                circle.shift(RIGHT * (j % 8) + UP * (j // 8))  # Simple grid layout for 8x8 input
                digit_img.add(circle)
            digit_img.scale(0.5).shift(RIGHT * i)
            digit_images.add(digit_img)

        # Show input digits
        self.play(LaggedStartMap(FadeIn, digit_images))
        self.wait(1)

        # Initialize the weight matrix as a grid
        weight_grid = VGroup()
        for i in range(input_size):
            for j in range(output_size):
                weight = DecimalNumber(weights[i, j], num_decimal_places=2)
                weight.move_to((i - input_size // 2, j - output_size // 2, 0))
                weight_grid.add(weight)

        self.play(Create(weight_grid))
        self.wait(1)

        # Show how weights update across epochs (due to L1 regularization)
        epochs = 5
        for epoch in range(epochs):
            gradients = np.random.uniform(-0.5, 0.5, (input_size, output_size))  # Simulated random gradients
            weights = l1_gradient_update(weights, gradients)  # Apply L1 update rule

            # Update the weight grid with new values
            for i, weight in enumerate(weight_grid):
                weight.set_value(weights[i % input_size, i // input_size])

            # Show current epoch
            epoch_text = Text(f"Epoch {epoch + 1}", font_size=24)
            epoch_text.to_edge(UP)
            self.play(Transform(weight_grid, weight_grid), Write(epoch_text))
            self.wait(0.5)
            self.play(FadeOut(epoch_text))

            # Emphasize sparse weights (L1) by shrinking to zero
            zeroed_weights = [weight for weight in weight_grid if abs(weight.get_value()) < 0.1]
            self.play(*[weight.set_opacity(0.5) for weight in zeroed_weights])
            self.wait(0.5)

        # Final message
        final_text = Text("L1 Regularization Creates Sparsity in Weights", font_size=24)
        final_text.to_edge(DOWN)
        self.play(Write(final_text))
        self.wait(2)

%manim -qm L1RegularizationWithMNIST

[11/11/24 00:55:09] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=440834;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=349428;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#158\158]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=778099;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=190101;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#164\164]8;;\
                             in your config file.                                                                  

[11/11/24 00:55:23] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=661289;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=139485;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#158\158]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=911557;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=75583;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#164\164]8;;\
                             in your config file.                                                                  

[11/11/24 00:55:30] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=463551;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=870083;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#158\158]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

...

[11/11/24 00:55:48] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=627566;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=751413;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#158\158]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=374681;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=381504;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#164\164]8;;\
                             in your config file.                                                                  

[11/11/24 00:55:53] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=687999;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=735026;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#158\158]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=65211;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=926855;file:///home/timothy_holdsworth/code/timholds.github.io-2/th-env/lib/python3.8/site-packages/manim/utils/hashing.py#164\164]8;;\
                             in your config file.                                                                  

TypeError: Unexpected argument DecimalNumber passed to Scene.play().